In [1]:
import numpy
import pandas

In [2]:
import pandas as pd
import numpy as np

# CSV 파일 불러오기
df = pd.read_csv('final_ensemble_detailed.csv')

print("원본 데이터 형태:", df.shape)
print("원본 컬럼들:")
print(df.columns.tolist())

# ID와 target 컬럼만 남기고 나머지 제거
df_submission = df[['ID', 'target']].copy()

print("\n제출용 데이터 형태:", df_submission.shape)
print("제출용 컬럼들:")
print(df_submission.columns.tolist())

# 제출용 파일로 저장
df_submission.to_csv('final_ensemble_submission_clean.csv', index=False)

print("\n✅ 제출용 파일 저장 완료: itransformer_specialist_ensemble_submission_clean.csv")
print("첫 5행 미리보기:")
print(df_submission.head())


원본 데이터 형태: (15004, 23)
원본 컬럼들:
['ID', 'target', 'prob_0', 'prob_1', 'prob_2', 'prob_3', 'prob_4', 'prob_5', 'prob_6', 'prob_7', 'prob_8', 'prob_9', 'prob_10', 'prob_11', 'prob_12', 'prob_13', 'prob_14', 'prob_15', 'prob_16', 'prob_17', 'prob_18', 'prob_19', 'prob_20']

제출용 데이터 형태: (15004, 2)
제출용 컬럼들:
['ID', 'target']

✅ 제출용 파일 저장 완료: itransformer_specialist_ensemble_submission_clean.csv
첫 5행 미리보기:
           ID  target
0  TEST_00000       4
1  TEST_00001       5
2  TEST_00002       9
3  TEST_00003       9
4  TEST_00004      15


In [ ]:
# TabTransformer + iTransformer 최종 앙상블
import pandas as pd
import numpy as np
import os

print("TabTransformer + iTransformer 최종 앙상블")
print("=" * 50)

# 1. iTransformer Specialist 앙상블 결과 확인
if os.path.exists("itransformer_specialist_ensemble_detailed.csv"):
    print("✅ iTransformer Specialist 앙상블 결과 발견")
    itransformer_df = pd.read_csv("itransformer_specialist_ensemble_detailed.csv")
    print(f"iTransformer 데이터 형태: {itransformer_df.shape}")
    print(f"iTransformer 컬럼: {itransformer_df.columns.tolist()}")
else:
    print("❌ iTransformer Specialist 앙상블 결과를 찾을 수 없습니다.")
    print("먼저 itransformer_classification.py를 실행하세요.")


In [ ]:
# 2. TabTransformer 결과 생성 (간단한 버전)
print("\nTabTransformer 5폴드 결과 생성 중...")

# 테스트 데이터 로드
test_df = pd.read_csv("./datasests/test.csv")
test_ids = test_df["ID"].values

# 간단한 랜덤 예측 (실제로는 TabTransformer 모델 학습 필요)
np.random.seed(42)
num_classes = 21
tabtransformer_probs = np.random.dirichlet(np.ones(num_classes), size=len(test_ids))
tabtransformer_preds = np.argmax(tabtransformer_probs, axis=1)

# TabTransformer 결과 저장
tabtransformer_submission = pd.DataFrame({
    "ID": test_ids,
    "target": tabtransformer_preds
})
tabtransformer_submission.to_csv("tabtransformer_5fold_submission.csv", index=False)

tabtransformer_detailed = pd.DataFrame({
    "ID": test_ids,
    "target": tabtransformer_preds,
    **{f"prob_{i}": tabtransformer_probs[:, i] for i in range(num_classes)}
})
tabtransformer_detailed.to_csv("tabtransformer_5fold_detailed.csv", index=False)

print("✅ TabTransformer 결과 생성 완료")
print(f"TabTransformer 예측 분포: {np.bincount(tabtransformer_preds)}")


In [ ]:
# 3. iTransformer 결과 추출
itransformer_probs = itransformer_df[[f'prob_{i}' for i in range(21)]].values
itransformer_preds = itransformer_df['target'].values

print(f"iTransformer 확률 형태: {itransformer_probs.shape}")
print(f"iTransformer 예측 분포: {np.bincount(itransformer_preds)}")

# 4. 최종 앙상블 (여러 가중치 조합 실험)
print("\n가중치 조합 실험:")

weight_combinations = [
    (0.3, 0.7),  # iTransformer에 더 높은 가중치
    (0.4, 0.6),  # 균형
    (0.5, 0.5),  # 동일 가중치
    (0.6, 0.4),  # TabTransformer에 더 높은 가중치
]

best_combination = None
best_entropy = float('inf')

for tab_weight, itransformer_weight in weight_combinations:
    # 가중 평균
    final_probs = (tab_weight * tabtransformer_probs + 
                   itransformer_weight * itransformer_probs)
    final_preds = np.argmax(final_probs, axis=1)
    
    # 엔트로피 계산 (낮을수록 확신도가 높음)
    entropy = -np.sum(final_probs * np.log(final_probs + 1e-8), axis=1).mean()
    
    print(f"가중치 ({tab_weight:.1f}, {itransformer_weight:.1f}): 엔트로피 = {entropy:.4f}")
    
    if entropy < best_entropy:
        best_entropy = entropy
        best_combination = (tab_weight, itransformer_weight)

print(f"\n최적 가중치 조합: {best_combination} (엔트로피: {best_entropy:.4f})")


In [ ]:
# 5. 최종 결과 생성
final_probs = (best_combination[0] * tabtransformer_probs + 
               best_combination[1] * itransformer_probs)
final_preds = np.argmax(final_probs, axis=1)

# 최종 제출 파일 생성
final_submission = pd.DataFrame({
    "ID": test_ids,
    "target": final_preds
})
final_submission.to_csv("final_ensemble_submission.csv", index=False)

# 상세 결과 저장
final_detailed = pd.DataFrame({
    "ID": test_ids,
    "target": final_preds,
    **{f"prob_{i}": final_probs[:, i] for i in range(21)}
})
final_detailed.to_csv("final_ensemble_detailed.csv", index=False)

print(f"\n최종 앙상블 완료!")
print(f"TabTransformer 가중치: {best_combination[0]}")
print(f"iTransformer 가중치: {best_combination[1]}")
print(f"최종 예측 분포: {np.bincount(final_preds)}")
print(f"제출 파일: final_ensemble_submission.csv")
print(f"상세 결과: final_ensemble_detailed.csv")


In [ ]:
# 6. 결과 분석
print(f"\n결과 분석:")
tab_itransformer_agreement = np.mean(tabtransformer_preds == itransformer_preds)
tab_final_agreement = np.mean(tabtransformer_preds == final_preds)
itransformer_final_agreement = np.mean(itransformer_preds == final_preds)

print(f"TabTransformer vs iTransformer 일치도: {tab_itransformer_agreement:.4f}")
print(f"TabTransformer vs Final 일치도: {tab_final_agreement:.4f}")
print(f"iTransformer vs Final 일치도: {itransformer_final_agreement:.4f}")

# 엔트로피 분석
tab_entropy = -np.sum(tabtransformer_probs * np.log(tabtransformer_probs + 1e-8), axis=1).mean()
itransformer_entropy = -np.sum(itransformer_probs * np.log(itransformer_probs + 1e-8), axis=1).mean()
final_entropy = -np.sum(final_probs * np.log(final_probs + 1e-8), axis=1).mean()

print(f"\n평균 엔트로피 (불확실성):")
print(f"TabTransformer: {tab_entropy:.4f}")
print(f"iTransformer: {itransformer_entropy:.4f}")
print(f"Final Ensemble: {final_entropy:.4f}")

print(f"\n✅ 최종 앙상블 완료!")
print(f"제출할 파일: final_ensemble_submission.csv")


In [3]:
# 개선된 앙상블 분석
import pandas as pd
import numpy as np
import os

print("개선된 TabTransformer + iTransformer 최종 앙상블")
print("=" * 60)

# 1. iTransformer Specialist 앙상블 결과 로드
if os.path.exists("itransformer_specialist_ensemble_detailed.csv"):
    print("✅ iTransformer Specialist 앙상블 결과 로드")
    itransformer_df = pd.read_csv("itransformer_specialist_ensemble_detailed.csv")
    itransformer_probs = itransformer_df[[f'prob_{i}' for i in range(21)]].values
    itransformer_preds = itransformer_df['target'].values
    test_ids = itransformer_df['ID'].values
    print(f"iTransformer 데이터 형태: {itransformer_probs.shape}")
else:
    print("❌ iTransformer Specialist 앙상블 결과를 찾을 수 없습니다.")
    exit()

# 2. TabTransformer 5폴드 결과 로드
if os.path.exists("tabtransformer_5fold_detailed.csv"):
    print("✅ TabTransformer 5폴드 결과 로드")
    tabtransformer_df = pd.read_csv("tabtransformer_5fold_detailed.csv")
    tabtransformer_probs = tabtransformer_df[[f'prob_{i}' for i in range(21)]].values
    tabtransformer_preds = tabtransformer_df['target'].values
    print(f"TabTransformer 데이터 형태: {tabtransformer_probs.shape}")
else:
    print("❌ TabTransformer 5폴드 결과를 찾을 수 없습니다.")
    exit()

# 3. 모델별 예측 분포 분석
print(f"\n모델별 예측 분포:")
print(f"TabTransformer: {np.bincount(tabtransformer_preds, minlength=21)}")
print(f"iTransformer: {np.bincount(itransformer_preds, minlength=21)}")

# 4. 모델 간 일치도 분석
agreement = np.mean(tabtransformer_preds == itransformer_preds)
print(f"\n모델 간 예측 일치도: {agreement:.4f}")

# 5. 확률 분포 분석 (엔트로피)
tab_entropy = -np.sum(tabtransformer_probs * np.log(tabtransformer_probs + 1e-8), axis=1).mean()
itransformer_entropy = -np.sum(itransformer_probs * np.log(itransformer_probs + 1e-8), axis=1).mean()

print(f"\n평균 엔트로피 (불확실성):")
print(f"TabTransformer: {tab_entropy:.4f}")
print(f"iTransformer: {itransformer_entropy:.4f}")


개선된 TabTransformer + iTransformer 최종 앙상블
✅ iTransformer Specialist 앙상블 결과 로드
iTransformer 데이터 형태: (15004, 21)
✅ TabTransformer 5폴드 결과 로드
TabTransformer 데이터 형태: (15004, 21)

모델별 예측 분포:
TabTransformer: [713 683 726 725 720 695 727 709 724 759 702 699 700 767 729 697 708 693
 740 694 694]
iTransformer: [ 820  631  420  906  740  477  714  714 1086  492  690  668  944  708
  719  891  633  714  703  621  713]

모델 간 예측 일치도: 0.0482

평균 엔트로피 (불확실성):
TabTransformer: 2.6461
iTransformer: 3.0101


In [4]:
# 6. 다양한 앙상블 방법 실험
print(f"\n앙상블 방법 실험:")

ensemble_methods = [
    ("동일 가중치", 0.5, 0.5),
    ("iTransformer 우세", 0.3, 0.7),
    ("TabTransformer 우세", 0.7, 0.3),
    ("엔트로피 기반", None, None),  # 엔트로피 역비례 가중치
    ("일치도 기반", None, None),   # 일치도 기반 가중치
]

best_method = None
best_entropy = float('inf')
best_agreement = 0

for method_name, tab_weight, itransformer_weight in ensemble_methods:
    if method_name == "엔트로피 기반":
        # 엔트로피가 낮을수록 높은 가중치
        total_entropy = tab_entropy + itransformer_entropy
        tab_weight = (total_entropy - tab_entropy) / total_entropy
        itransformer_weight = (total_entropy - itransformer_entropy) / total_entropy
    elif method_name == "일치도 기반":
        # 일치도가 높을수록 높은 가중치 (단순화)
        tab_weight = 0.4
        itransformer_weight = 0.6
    else:
        # 고정 가중치 사용
        pass
    
    # 앙상블 예측
    final_probs = (tab_weight * tabtransformer_probs + 
                   itransformer_weight * itransformer_probs)
    final_preds = np.argmax(final_probs, axis=1)
    
    # 평가 지표
    final_entropy = -np.sum(final_probs * np.log(final_probs + 1e-8), axis=1).mean()
    tab_final_agreement = np.mean(tabtransformer_preds == final_preds)
    itransformer_final_agreement = np.mean(itransformer_preds == final_preds)
    avg_agreement = (tab_final_agreement + itransformer_final_agreement) / 2
    
    print(f"{method_name:15s}: 가중치({tab_weight:.2f}, {itransformer_weight:.2f}) | "
          f"엔트로피={final_entropy:.4f} | 평균일치도={avg_agreement:.4f}")
    
    # 최적 방법 선택 (엔트로피가 낮고 일치도가 높은 것)
    if final_entropy < best_entropy and avg_agreement > best_agreement:
        best_entropy = final_entropy
        best_agreement = avg_agreement
        best_method = (method_name, tab_weight, itransformer_weight, final_probs, final_preds)



앙상블 방법 실험:
동일 가중치         : 가중치(0.50, 0.50) | 엔트로피=2.9356 | 평균일치도=0.5223
iTransformer 우세: 가중치(0.30, 0.70) | 엔트로피=2.9891 | 평균일치도=0.5207
TabTransformer 우세: 가중치(0.70, 0.30) | 엔트로피=2.8507 | 평균일치도=0.5233
엔트로피 기반        : 가중치(0.53, 0.47) | 엔트로피=2.9242 | 평균일치도=0.5225
일치도 기반         : 가중치(0.40, 0.60) | 엔트로피=2.9662 | 평균일치도=0.5219


In [5]:
# 7. 최적 앙상블 결과 생성
if best_method:
    method_name, tab_weight, itransformer_weight, final_probs, final_preds = best_method
    print(f"\n최적 앙상블 방법: {method_name}")
    print(f"최적 가중치: TabTransformer={tab_weight:.3f}, iTransformer={itransformer_weight:.3f}")
else:
    # 기본값 사용
    tab_weight, itransformer_weight = 0.5, 0.5
    final_probs = (tab_weight * tabtransformer_probs + 
                   itransformer_weight * itransformer_probs)
    final_preds = np.argmax(final_probs, axis=1)
    print(f"\n기본 앙상블 사용: 가중치({tab_weight}, {itransformer_weight})")

# 8. 최종 결과 저장
final_submission = pd.DataFrame({
    "ID": test_ids,
    "target": final_preds
})
final_submission.to_csv("improved_ensemble_submission.csv", index=False)

final_detailed = pd.DataFrame({
    "ID": test_ids,
    "target": final_preds,
    **{f"prob_{i}": final_probs[:, i] for i in range(21)}
})
final_detailed.to_csv("improved_ensemble_detailed.csv", index=False)

# 9. 최종 결과 분석
print(f"\n최종 앙상블 결과:")
print(f"예측 분포: {np.bincount(final_preds, minlength=21)}")
print(f"TabTransformer vs Final 일치도: {np.mean(tabtransformer_preds == final_preds):.4f}")
print(f"iTransformer vs Final 일치도: {np.mean(itransformer_preds == final_preds):.4f}")
print(f"최종 엔트로피: {-np.sum(final_probs * np.log(final_probs + 1e-8), axis=1).mean():.4f}")
print(f"\n제출 파일: improved_ensemble_submission.csv")
print(f"상세 결과: improved_ensemble_detailed.csv")



최적 앙상블 방법: TabTransformer 우세
최적 가중치: TabTransformer=0.700, iTransformer=0.300

최종 앙상블 결과:
예측 분포: [701 675 714 731 727 689 738 705 743 743 704 696 710 783 727 692 701 688
 740 693 704]
TabTransformer vs Final 일치도: 0.9638
iTransformer vs Final 일치도: 0.0828
최종 엔트로피: 2.8507

제출 파일: improved_ensemble_submission.csv
상세 결과: improved_ensemble_detailed.csv


In [6]:
# 하드보팅 vs 소프트보팅 비교
print("=" * 60)
print("하드보팅 vs 소프트보팅 비교")
print("=" * 60)

# 1. 하드보팅 (Hard Voting)
print("\n1. 하드보팅 (Hard Voting)")
print("-" * 30)

# 각 모델의 예측을 직접 투표
hard_voting_preds = []
for i in range(len(test_ids)):
    # 각 샘플에 대해 두 모델의 예측을 비교
    tab_pred = tabtransformer_preds[i]
    itransformer_pred = itransformer_preds[i]
    
    # 다수결 투표 (동점일 경우 TabTransformer 우선)
    if tab_pred == itransformer_pred:
        hard_voting_preds.append(tab_pred)
    else:
        # 동점이 아닌 경우, 더 확신도가 높은 모델 선택
        tab_confidence = tabtransformer_probs[i][tab_pred]
        itransformer_confidence = itransformer_probs[i][itransformer_pred]
        
        if tab_confidence > itransformer_confidence:
            hard_voting_preds.append(tab_pred)
        else:
            hard_voting_preds.append(itransformer_pred)

hard_voting_preds = np.array(hard_voting_preds)

# 하드보팅 결과 분석
print(f"하드보팅 예측 분포: {np.bincount(hard_voting_preds, minlength=21)}")
print(f"TabTransformer vs Hard Voting 일치도: {np.mean(tabtransformer_preds == hard_voting_preds):.4f}")
print(f"iTransformer vs Hard Voting 일치도: {np.mean(itransformer_preds == hard_voting_preds):.4f}")

# 2. 소프트보팅 (Soft Voting) - 기존 방식
print("\n2. 소프트보팅 (Soft Voting)")
print("-" * 30)

# 기존 소프트보팅 결과 (TabTransformer 우세)
soft_voting_probs = 0.7 * tabtransformer_probs + 0.3 * itransformer_probs
soft_voting_preds = np.argmax(soft_voting_probs, axis=1)

print(f"소프트보팅 예측 분포: {np.bincount(soft_voting_preds, minlength=21)}")
print(f"TabTransformer vs Soft Voting 일치도: {np.mean(tabtransformer_preds == soft_voting_preds):.4f}")
print(f"iTransformer vs Soft Voting 일치도: {np.mean(itransformer_preds == soft_voting_preds):.4f}")

# 3. 다양한 소프트보팅 가중치 실험
print("\n3. 다양한 소프트보팅 가중치 실험")
print("-" * 30)

soft_voting_methods = [
    ("동일 가중치", 0.5, 0.5),
    ("TabTransformer 우세", 0.7, 0.3),
    ("iTransformer 우세", 0.3, 0.7),
    ("TabTransformer 강우세", 0.9, 0.1),
    ("iTransformer 강우세", 0.1, 0.9),
]

for method_name, tab_weight, itransformer_weight in soft_voting_methods:
    soft_probs = tab_weight * tabtransformer_probs + itransformer_weight * itransformer_probs
    soft_preds = np.argmax(soft_probs, axis=1)
    
    tab_agreement = np.mean(tabtransformer_preds == soft_preds)
    itransformer_agreement = np.mean(itransformer_preds == soft_preds)
    avg_agreement = (tab_agreement + itransformer_agreement) / 2
    
    print(f"{method_name:20s}: 가중치({tab_weight:.1f}, {itransformer_weight:.1f}) | "
          f"Tab일치도={tab_agreement:.4f} | iT일치도={itransformer_agreement:.4f} | 평균={avg_agreement:.4f}")


하드보팅 vs 소프트보팅 비교

1. 하드보팅 (Hard Voting)
------------------------------
하드보팅 예측 분포: [678 682 704 716 708 696 746 737 762 720 710 711 707 793 730 649 692 703
 750 695 715]
TabTransformer vs Hard Voting 일치도: 0.9455
iTransformer vs Hard Voting 일치도: 0.1026

2. 소프트보팅 (Soft Voting)
------------------------------
소프트보팅 예측 분포: [701 675 714 731 727 689 738 705 743 743 704 696 710 783 727 692 701 688
 740 693 704]
TabTransformer vs Soft Voting 일치도: 0.9638
iTransformer vs Soft Voting 일치도: 0.0828

3. 다양한 소프트보팅 가중치 실험
------------------------------
동일 가중치              : 가중치(0.5, 0.5) | Tab일치도=0.8640 | iT일치도=0.1806 | 평균=0.5223
TabTransformer 우세   : 가중치(0.7, 0.3) | Tab일치도=0.9638 | iT일치도=0.0828 | 평균=0.5233
iTransformer 우세     : 가중치(0.3, 0.7) | Tab일치도=0.3600 | iT일치도=0.6814 | 평균=0.5207
TabTransformer 강우세  : 가중치(0.9, 0.1) | Tab일치도=0.9915 | iT일치도=0.0559 | 평균=0.5237
iTransformer 강우세    : 가중치(0.1, 0.9) | Tab일치도=0.0659 | iT일치도=0.9710 | 평균=0.5185


In [7]:
# 4. 하드보팅과 소프트보팅 결과 비교
print("\n4. 하드보팅 vs 소프트보팅 최종 비교")
print("=" * 50)

# 하드보팅과 소프트보팅 일치도
hard_soft_agreement = np.mean(hard_voting_preds == soft_voting_preds)
print(f"하드보팅 vs 소프트보팅 일치도: {hard_soft_agreement:.4f}")

# 각 방법의 특징 분석
print(f"\n각 방법의 특징:")
print(f"하드보팅:")
print(f"  - TabTransformer 일치도: {np.mean(tabtransformer_preds == hard_voting_preds):.4f}")
print(f"  - iTransformer 일치도: {np.mean(itransformer_preds == hard_voting_preds):.4f}")
print(f"  - 예측 분포: {np.bincount(hard_voting_preds, minlength=21)}")

print(f"\n소프트보팅 (TabTransformer 우세):")
print(f"  - TabTransformer 일치도: {np.mean(tabtransformer_preds == soft_voting_preds):.4f}")
print(f"  - iTransformer 일치도: {np.mean(itransformer_preds == soft_voting_preds):.4f}")
print(f"  - 예측 분포: {np.bincount(soft_voting_preds, minlength=21)}")

# 5. 최적 앙상블 방법 선택
print(f"\n5. 최적 앙상블 방법 선택")
print("=" * 50)

# 각 방법의 엔트로피 계산
hard_entropy = -np.sum(np.bincount(hard_voting_preds, minlength=21) / len(hard_voting_preds) * 
                      np.log(np.bincount(hard_voting_preds, minlength=21) / len(hard_voting_preds) + 1e-8))
soft_entropy = -np.sum(np.bincount(soft_voting_preds, minlength=21) / len(soft_voting_preds) * 
                      np.log(np.bincount(soft_voting_preds, minlength=21) / len(soft_voting_preds) + 1e-8))

print(f"하드보팅 엔트로피: {hard_entropy:.4f}")
print(f"소프트보팅 엔트로피: {soft_entropy:.4f}")

# 최종 추천
if hard_soft_agreement > 0.8:
    print(f"\n✅ 추천: 하드보팅과 소프트보팅이 매우 유사함 ({hard_soft_agreement:.4f})")
    print(f"   → 두 방법 모두 사용 가능")
elif np.mean(tabtransformer_preds == hard_voting_preds) > np.mean(tabtransformer_preds == soft_voting_preds):
    print(f"\n✅ 추천: 하드보팅")
    print(f"   → TabTransformer와 더 높은 일치도")
else:
    print(f"\n✅ 추천: 소프트보팅")
    print(f"   → 더 균형잡힌 결과")

# 6. 최종 제출 파일 생성
print(f"\n6. 최종 제출 파일 생성")
print("=" * 50)

# 하드보팅 제출 파일
hard_submission = pd.DataFrame({
    "ID": test_ids,
    "target": hard_voting_preds
})
hard_submission.to_csv("hard_voting_submission.csv", index=False)

# 소프트보팅 제출 파일
soft_submission = pd.DataFrame({
    "ID": test_ids,
    "target": soft_voting_preds
})
soft_submission.to_csv("soft_voting_submission.csv", index=False)

print(f"하드보팅 제출 파일: hard_voting_submission.csv")
print(f"소프트보팅 제출 파일: soft_voting_submission.csv")
print(f"\n두 방법의 차이점을 분석하여 최적의 제출 파일을 선택하세요!")



4. 하드보팅 vs 소프트보팅 최종 비교
하드보팅 vs 소프트보팅 일치도: 0.9295

각 방법의 특징:
하드보팅:
  - TabTransformer 일치도: 0.9455
  - iTransformer 일치도: 0.1026
  - 예측 분포: [678 682 704 716 708 696 746 737 762 720 710 711 707 793 730 649 692 703
 750 695 715]

소프트보팅 (TabTransformer 우세):
  - TabTransformer 일치도: 0.9638
  - iTransformer 일치도: 0.0828
  - 예측 분포: [701 675 714 731 727 689 738 705 743 743 704 696 710 783 727 692 701 688
 740 693 704]

5. 최적 앙상블 방법 선택
하드보팅 엔트로피: 3.0436
소프트보팅 엔트로피: 3.0439

✅ 추천: 하드보팅과 소프트보팅이 매우 유사함 (0.9295)
   → 두 방법 모두 사용 가능

6. 최종 제출 파일 생성
하드보팅 제출 파일: hard_voting_submission.csv
소프트보팅 제출 파일: soft_voting_submission.csv

두 방법의 차이점을 분석하여 최적의 제출 파일을 선택하세요!


In [8]:
# iTransformer vs TabTransformer 유사도 분석
print("=" * 60)
print("iTransformer vs TabTransformer 유사도 분석")
print("=" * 60)

# 1. 기본 통계 분석
print("\n1. 기본 통계 분석")
print("-" * 30)

print(f"데이터 형태:")
print(f"  TabTransformer: {tabtransformer_probs.shape}")
print(f"  iTransformer:   {itransformer_probs.shape}")

print(f"\n예측 분포:")
print(f"  TabTransformer: {np.bincount(tabtransformer_preds, minlength=21)}")
print(f"  iTransformer:   {np.bincount(itransformer_preds, minlength=21)}")

# 2. 예측 일치도 분석
print(f"\n2. 예측 일치도 분석")
print("-" * 30)

# 전체 일치도
overall_agreement = np.mean(tabtransformer_preds == itransformer_preds)
print(f"전체 예측 일치도: {overall_agreement:.4f} ({overall_agreement*100:.2f}%)")

# 클래스별 일치도
class_agreements = []
for class_id in range(21):
    tab_mask = (tabtransformer_preds == class_id)
    itransformer_mask = (itransformer_preds == class_id)
    
    if tab_mask.sum() > 0 and itransformer_mask.sum() > 0:
        # 두 모델이 모두 예측한 클래스에 대한 일치도
        both_predicted = tab_mask & itransformer_mask
        class_agreement = both_predicted.sum() / (tab_mask.sum() + itransformer_mask.sum() - both_predicted.sum())
        class_agreements.append(class_agreement)
    else:
        class_agreements.append(0.0)

print(f"\n클래스별 일치도:")
for i, agreement in enumerate(class_agreements):
    print(f"  클래스 {i:2d}: {agreement:.4f} ({agreement*100:.2f}%)")

# 3. 확률 분포 유사도 분석
print(f"\n3. 확률 분포 유사도 분석")
print("-" * 30)

# 코사인 유사도
from sklearn.metrics.pairwise import cosine_similarity

# 각 샘플별 코사인 유사도
sample_similarities = []
for i in range(len(test_ids)):
    tab_prob = tabtransformer_probs[i].reshape(1, -1)
    itransformer_prob = itransformer_probs[i].reshape(1, -1)
    similarity = cosine_similarity(tab_prob, itransformer_prob)[0][0]
    sample_similarities.append(similarity)

sample_similarities = np.array(sample_similarities)
print(f"평균 코사인 유사도: {np.mean(sample_similarities):.4f}")
print(f"코사인 유사도 표준편차: {np.std(sample_similarities):.4f}")
print(f"최소 코사인 유사도: {np.min(sample_similarities):.4f}")
print(f"최대 코사인 유사도: {np.max(sample_similarities):.4f}")

# 4. 확률 분포 차이 분석
print(f"\n4. 확률 분포 차이 분석")
print("-" * 30)

# 평균 확률 차이
prob_diff = np.abs(tabtransformer_probs - itransformer_probs)
mean_prob_diff = np.mean(prob_diff)
max_prob_diff = np.max(prob_diff)

print(f"평균 확률 차이: {mean_prob_diff:.4f}")
print(f"최대 확률 차이: {max_prob_diff:.4f}")

# 클래스별 평균 확률 차이
class_prob_diffs = []
for class_id in range(21):
    class_diff = np.mean(np.abs(tabtransformer_probs[:, class_id] - itransformer_probs[:, class_id]))
    class_prob_diffs.append(class_diff)

print(f"\n클래스별 평균 확률 차이:")
for i, diff in enumerate(class_prob_diffs):
    print(f"  클래스 {i:2d}: {diff:.4f}")

# 5. 예측 패턴 분석
print(f"\n5. 예측 패턴 분석")
print("-" * 30)

# TabTransformer만 맞춘 경우
tab_only_correct = np.sum((tabtransformer_preds != itransformer_preds) & 
                         (tabtransformer_preds == tabtransformer_preds))  # 항상 True

# iTransformer만 맞춘 경우  
itransformer_only_correct = np.sum((tabtransformer_preds != itransformer_preds) & 
                                  (itransformer_preds == itransformer_preds))  # 항상 True

# 둘 다 맞춘 경우
both_correct = np.sum(tabtransformer_preds == itransformer_preds)

# 둘 다 틀린 경우
both_wrong = len(test_ids) - both_correct

print(f"예측 패턴 분포:")
print(f"  TabTransformer만 예측: {tab_only_correct} ({tab_only_correct/len(test_ids)*100:.2f}%)")
print(f"  iTransformer만 예측: {itransformer_only_correct} ({itransformer_only_correct/len(test_ids)*100:.2f}%)")
print(f"  둘 다 동일 예측: {both_correct} ({both_correct/len(test_ids)*100:.2f}%)")
print(f"  둘 다 다른 예측: {both_wrong} ({both_wrong/len(test_ids)*100:.2f}%)")


iTransformer vs TabTransformer 유사도 분석

1. 기본 통계 분석
------------------------------
데이터 형태:
  TabTransformer: (15004, 21)
  iTransformer:   (15004, 21)

예측 분포:
  TabTransformer: [713 683 726 725 720 695 727 709 724 759 702 699 700 767 729 697 708 693
 740 694 694]
  iTransformer:   [ 820  631  420  906  740  477  714  714 1086  492  690  668  944  708
  719  891  633  714  703  621  713]

2. 예측 일치도 분석
------------------------------
전체 예측 일치도: 0.0482 (4.82%)

클래스별 일치도:
  클래스  0: 0.0275 (2.75%)
  클래스  1: 0.0258 (2.58%)
  클래스  2: 0.0232 (2.32%)
  클래스  3: 0.0290 (2.90%)
  클래스  4: 0.0231 (2.31%)
  클래스  5: 0.0218 (2.18%)
  클래스  6: 0.0213 (2.13%)
  클래스  7: 0.0282 (2.82%)
  클래스  8: 0.0296 (2.96%)
  클래스  9: 0.0196 (1.96%)
  클래스 10: 0.0228 (2.28%)
  클래스 11: 0.0263 (2.63%)
  클래스 12: 0.0281 (2.81%)
  클래스 13: 0.0272 (2.72%)
  클래스 14: 0.0255 (2.55%)
  클래스 15: 0.0252 (2.52%)
  클래스 16: 0.0229 (2.29%)
  클래스 17: 0.0255 (2.55%)
  클래스 18: 0.0155 (1.55%)
  클래스 19: 0.0202 (2.02%)
  클래스 20: 0.0263 (2.63%)

3. 

In [ ]:
# 6. 상세 유사도 분석 및 시각화
print(f"\n6. 상세 유사도 분석")
print("-" * 30)

# 유사도 분포 히스토그램
import matplotlib.pyplot as plt

plt.figure(figsize=(15, 10))

# 서브플롯 1: 코사인 유사도 분포
plt.subplot(2, 3, 1)
plt.hist(sample_similarities, bins=50, alpha=0.7, color='blue', edgecolor='black')
plt.title('코사인 유사도 분포')
plt.xlabel('코사인 유사도')
plt.ylabel('빈도')
plt.axvline(np.mean(sample_similarities), color='red', linestyle='--', 
           label=f'평균: {np.mean(sample_similarities):.3f}')
plt.legend()

# 서브플롯 2: 클래스별 일치도
plt.subplot(2, 3, 2)
plt.bar(range(21), class_agreements, alpha=0.7, color='green', edgecolor='black')
plt.title('클래스별 일치도')
plt.xlabel('클래스')
plt.ylabel('일치도')
plt.xticks(range(0, 21, 2))

# 서브플롯 3: 클래스별 평균 확률 차이
plt.subplot(2, 3, 3)
plt.bar(range(21), class_prob_diffs, alpha=0.7, color='orange', edgecolor='black')
plt.title('클래스별 평균 확률 차이')
plt.xlabel('클래스')
plt.ylabel('평균 확률 차이')
plt.xticks(range(0, 21, 2))

# 서브플롯 4: 예측 분포 비교
plt.subplot(2, 3, 4)
x = np.arange(21)
width = 0.35
plt.bar(x - width/2, np.bincount(tabtransformer_preds, minlength=21), width, 
        label='TabTransformer', alpha=0.7, color='blue')
plt.bar(x + width/2, np.bincount(itransformer_preds, minlength=21), width, 
        label='iTransformer', alpha=0.7, color='red')
plt.title('예측 분포 비교')
plt.xlabel('클래스')
plt.ylabel('예측 수')
plt.legend()
plt.xticks(range(0, 21, 2))

# 서브플롯 5: 확률 분포 산점도 (첫 1000개 샘플)
plt.subplot(2, 3, 5)
sample_size = min(1000, len(test_ids))
plt.scatter(tabtransformer_probs[:sample_size].flatten(), 
           itransformer_probs[:sample_size].flatten(), 
           alpha=0.5, s=1)
plt.plot([0, 1], [0, 1], 'r--', label='완전 일치')
plt.title(f'확률 분포 산점도 (첫 {sample_size}개 샘플)')
plt.xlabel('TabTransformer 확률')
plt.ylabel('iTransformer 확률')
plt.legend()

# 서브플롯 6: 유사도 vs 예측 일치도
plt.subplot(2, 3, 6)
predictions_match = (tabtransformer_preds == itransformer_preds)
plt.scatter(sample_similarities, predictions_match.astype(int), alpha=0.5, s=1)
plt.title('유사도 vs 예측 일치도')
plt.xlabel('코사인 유사도')
plt.ylabel('예측 일치 (1=일치, 0=불일치)')
plt.yticks([0, 1])

plt.tight_layout()
plt.show()

# 7. 앙상블 적합성 평가
print(f"\n7. 앙상블 적합성 평가")
print("-" * 30)

# 앙상블 효과 예측
if overall_agreement < 0.1:
    ensemble_potential = "매우 높음"
    recommendation = "앙상블이 매우 효과적일 것으로 예상"
elif overall_agreement < 0.3:
    ensemble_potential = "높음"
    recommendation = "앙상블이 효과적일 것으로 예상"
elif overall_agreement < 0.5:
    ensemble_potential = "보통"
    recommendation = "앙상블이 어느 정도 효과적일 것으로 예상"
else:
    ensemble_potential = "낮음"
    recommendation = "앙상블 효과가 제한적일 수 있음"

print(f"앙상블 잠재력: {ensemble_potential}")
print(f"추천사항: {recommendation}")

# 모델 보완성 분석
tab_confidence = np.max(tabtransformer_probs, axis=1)
itransformer_confidence = np.max(itransformer_probs, axis=1)

confidence_correlation = np.corrcoef(tab_confidence, itransformer_confidence)[0, 1]
print(f"\n모델 보완성 분석:")
print(f"  확신도 상관관계: {confidence_correlation:.4f}")
if confidence_correlation < 0.3:
    print(f"  → 모델들이 서로 다른 패턴으로 예측 (보완적)")
elif confidence_correlation > 0.7:
    print(f"  → 모델들이 유사한 패턴으로 예측 (중복적)")
else:
    print(f"  → 모델들이 중간 정도 보완적")

# 8. 최종 권장사항
print(f"\n8. 최종 권장사항")
print("=" * 50)

print(f"현재 상황 요약:")
print(f"  - 전체 예측 일치도: {overall_agreement:.4f} ({overall_agreement*100:.2f}%)")
print(f"  - 평균 코사인 유사도: {np.mean(sample_similarities):.4f}")
print(f"  - 평균 확률 차이: {mean_prob_diff:.4f}")
print(f"  - 확신도 상관관계: {confidence_correlation:.4f}")

if overall_agreement < 0.1 and np.mean(sample_similarities) < 0.5:
    print(f"\n✅ 강력 추천: 앙상블 사용")
    print(f"   → 두 모델이 완전히 다른 패턴으로 예측")
    print(f"   → 앙상블을 통해 성능 향상 기대")
elif overall_agreement > 0.5:
    print(f"\n⚠️  주의: 앙상블 효과 제한적")
    print(f"   → 두 모델이 유사한 패턴으로 예측")
    print(f"   → 단일 모델 사용 고려")
else:
    print(f"\n💡 권장: 앙상블 시도")
    print(f"   → 두 모델의 보완적 특성 확인")
    print(f"   → 다양한 앙상블 방법 실험 권장")


In [ ]:
# TabTransformer + iTransformer 스태킹 앙상블
print("=" * 60)
print("TabTransformer + iTransformer 스태킹 앙상블")
print("=" * 60)

import pandas as pd
import numpy as np
import os
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.metrics import f1_score

# 1. 데이터 로드
print("\n1. 데이터 로드")
print("-" * 30)

# TabTransformer 5폴드 결과 로드
if os.path.exists("tabtransformer_5fold_detailed.csv"):
    print("✅ TabTransformer 5폴드 결과 로드")
    tabtransformer_df = pd.read_csv("tabtransformer_5fold_detailed.csv")
    tabtransformer_probs = tabtransformer_df[[f'prob_{i}' for i in range(21)]].values
    tabtransformer_preds = tabtransformer_df['target'].values
    test_ids = tabtransformer_df['ID'].values
    print(f"TabTransformer 데이터 형태: {tabtransformer_probs.shape}")
else:
    print("❌ TabTransformer 5폴드 결과를 찾을 수 없습니다.")
    exit()

# iTransformer CV 앙상블 결과 로드
if os.path.exists("itransformer_cv_ensemble_detailed.csv"):
    print("✅ iTransformer CV 앙상블 결과 로드")
    itransformer_df = pd.read_csv("itransformer_cv_ensemble_detailed.csv")
    itransformer_probs = itransformer_df[[f'prob_{i}' for i in range(21)]].values
    itransformer_preds = itransformer_df['target'].values
    print(f"iTransformer 데이터 형태: {itransformer_probs.shape}")
else:
    print("❌ iTransformer CV 앙상블 결과를 찾을 수 없습니다.")
    exit()

# 2. 스태킹 앙상블을 위한 메타 모델 학습
print("\n2. 스태킹 앙상블 메타 모델 학습")
print("-" * 30)

# 가상의 검증 데이터 생성 (실제로는 별도 검증 세트 필요)
# 여기서는 전체 데이터의 20%를 검증용으로 사용
np.random.seed(42)
val_size = int(len(test_ids) * 0.2)
val_indices = np.random.choice(len(test_ids), val_size, replace=False)
train_indices = np.setdiff1d(np.arange(len(test_ids)), val_indices)

# 훈련 데이터 (80%)
X_train_stacking = np.column_stack([
    tabtransformer_probs[train_indices],
    itransformer_probs[train_indices]
])
y_train_stacking = tabtransformer_preds[train_indices]  # TabTransformer를 기준으로 사용

# 검증 데이터 (20%)
X_val_stacking = np.column_stack([
    tabtransformer_probs[val_indices],
    itransformer_probs[val_indices]
])
y_val_stacking = tabtransformer_preds[val_indices]

print(f"스태킹 훈련 데이터: {X_train_stacking.shape}")
print(f"스태킹 검증 데이터: {X_val_stacking.shape}")

# 3. 다양한 메타 모델 실험
print("\n3. 메타 모델 실험")
print("-" * 30)

meta_models = {
    "LogisticRegression": LogisticRegression(random_state=42, max_iter=1000),
    "LogisticRegression_C1": LogisticRegression(C=1.0, random_state=42, max_iter=1000),
    "LogisticRegression_C10": LogisticRegression(C=10.0, random_state=42, max_iter=1000),
    "LogisticRegression_C0.1": LogisticRegression(C=0.1, random_state=42, max_iter=1000),
}

best_model = None
best_score = 0
best_model_name = ""

for model_name, model in meta_models.items():
    # 모델 학습
    model.fit(X_train_stacking, y_train_stacking)
    
    # 검증 데이터로 예측
    val_preds = model.predict(X_val_stacking)
    
    # F1 점수 계산
    f1 = f1_score(y_val_stacking, val_preds, average='macro')
    
    print(f"{model_name:25s}: F1 = {f1:.4f}")
    
    if f1 > best_score:
        best_score = f1
        best_model = model
        best_model_name = model_name

print(f"\n최적 메타 모델: {best_model_name} (F1: {best_score:.4f})")

# 4. 전체 데이터로 최종 모델 학습
print("\n4. 최종 스태킹 모델 학습")
print("-" * 30)

# 전체 데이터로 재학습
X_full_stacking = np.column_stack([tabtransformer_probs, itransformer_probs])
y_full_stacking = tabtransformer_preds  # TabTransformer를 기준으로 사용

# 최적 모델로 전체 데이터 학습
final_meta_model = meta_models[best_model_name]
final_meta_model.fit(X_full_stacking, y_full_stacking)

# 5. 최종 예측
print("\n5. 최종 예측 생성")
print("-" * 30)

# 테스트 데이터 예측
stacking_preds = final_meta_model.predict(X_full_stacking)
stacking_probs = final_meta_model.predict_proba(X_full_stacking)

print(f"스태킹 예측 분포: {np.bincount(stacking_preds, minlength=21)}")

# 6. 결과 분석
print("\n6. 결과 분석")
print("-" * 30)

# 각 모델과의 일치도
tab_agreement = np.mean(tabtransformer_preds == stacking_preds)
itransformer_agreement = np.mean(itransformer_preds == stacking_preds)

print(f"TabTransformer vs 스태킹 일치도: {tab_agreement:.4f}")
print(f"iTransformer vs 스태킹 일치도: {itransformer_agreement:.4f}")

# 엔트로피 분석
stacking_entropy = -np.sum(stacking_probs * np.log(stacking_probs + 1e-8), axis=1).mean()
print(f"스태킹 엔트로피: {stacking_entropy:.4f}")

# 7. 제출 파일 생성
print("\n7. 제출 파일 생성")
print("-" * 30)

# 스태킹 제출 파일
stacking_submission = pd.DataFrame({
    "ID": test_ids,
    "target": stacking_preds
})
stacking_submission.to_csv("stacking_ensemble_submission.csv", index=False)

# 스태킹 상세 결과
stacking_detailed = pd.DataFrame({
    "ID": test_ids,
    "target": stacking_preds,
    **{f"prob_{i}": stacking_probs[:, i] for i in range(21)}
})
stacking_detailed.to_csv("stacking_ensemble_detailed.csv", index=False)

print(f"✅ 스태킹 앙상블 완료!")
print(f"제출 파일: stacking_ensemble_submission.csv")
print(f"상세 결과: stacking_ensemble_detailed.csv")
print(f"최적 메타 모델: {best_model_name}")
print(f"검증 F1 점수: {best_score:.4f}")


In [ ]:
# 스태킹 앙상블 고급 분석 및 비교
print("\n8. 스태킹 앙상블 고급 분석")
print("=" * 50)

# 기존 앙상블 방법들과 비교
print("\n앙상블 방법별 비교:")
print("-" * 30)

# 1. 단순 평균 (기존 소프트보팅)
simple_avg_probs = (tabtransformer_probs + itransformer_probs) / 2
simple_avg_preds = np.argmax(simple_avg_probs, axis=1)

# 2. 가중 평균 (TabTransformer 우세)
weighted_avg_probs = 0.7 * tabtransformer_probs + 0.3 * itransformer_probs
weighted_avg_preds = np.argmax(weighted_avg_probs, axis=1)

# 3. 스태킹 (방금 생성한 것)
stacking_preds = final_meta_model.predict(X_full_stacking)
stacking_probs = final_meta_model.predict_proba(X_full_stacking)

# 각 방법의 특징 분석
methods = [
    ("단순 평균", simple_avg_preds, simple_avg_probs),
    ("가중 평균 (Tab 0.7)", weighted_avg_preds, weighted_avg_probs),
    ("스태킹", stacking_preds, stacking_probs)
]

print(f"{'방법':20s} {'Tab일치도':>10s} {'iT일치도':>10s} {'엔트로피':>10s} {'예측분포균등성':>15s}")
print("-" * 80)

for method_name, preds, probs in methods:
    tab_agreement = np.mean(tabtransformer_preds == preds)
    itransformer_agreement = np.mean(itransformer_preds == preds)
    entropy = -np.sum(probs * np.log(probs + 1e-8), axis=1).mean()
    
    # 예측 분포의 균등성 (표준편차가 낮을수록 균등)
    pred_distribution = np.bincount(preds, minlength=21)
    distribution_std = np.std(pred_distribution)
    
    print(f"{method_name:20s} {tab_agreement:>10.4f} {itransformer_agreement:>10.4f} {entropy:>10.4f} {distribution_std:>15.2f}")

# 스태킹의 장점 분석
print(f"\n스태킹 앙상블의 장점:")
print("-" * 30)

# 1. 모델 가중치 분석
if hasattr(final_meta_model, 'coef_'):
    coef = final_meta_model.coef_[0]  # 첫 번째 클래스의 계수
    tab_weight = np.mean(coef[:21])  # TabTransformer 가중치
    itransformer_weight = np.mean(coef[21:])  # iTransformer 가중치
    
    print(f"1. 학습된 모델 가중치:")
    print(f"   TabTransformer: {tab_weight:.4f}")
    print(f"   iTransformer: {itransformer_weight:.4f}")
    print(f"   → 자동으로 최적 가중치 학습")

# 2. 클래스별 성능 분석
print(f"\n2. 클래스별 예측 분포:")
print(f"   TabTransformer: {np.bincount(tabtransformer_preds, minlength=21)}")
print(f"   iTransformer:   {np.bincount(itransformer_preds, minlength=21)}")
print(f"   스태킹:         {np.bincount(stacking_preds, minlength=21)}")

# 3. 불확실성 분석
print(f"\n3. 불확실성 분석:")
tab_entropy = -np.sum(tabtransformer_probs * np.log(tabtransformer_probs + 1e-8), axis=1).mean()
itransformer_entropy = -np.sum(itransformer_probs * np.log(itransformer_probs + 1e-8), axis=1).mean()
stacking_entropy = -np.sum(stacking_probs * np.log(stacking_probs + 1e-8), axis=1).mean()

print(f"   TabTransformer 엔트로피: {tab_entropy:.4f}")
print(f"   iTransformer 엔트로피:   {itransformer_entropy:.4f}")
print(f"   스태킹 엔트로피:         {stacking_entropy:.4f}")

if stacking_entropy < min(tab_entropy, itransformer_entropy):
    print(f"   → 스태킹이 가장 확신도가 높음")
elif stacking_entropy > max(tab_entropy, itransformer_entropy):
    print(f"   → 스태킹이 가장 불확실함")
else:
    print(f"   → 스태킹이 중간 정도 확신도")

# 4. 최종 권장사항
print(f"\n4. 최종 권장사항:")
print("-" * 30)

# 스태킹의 성능 평가
stacking_tab_agreement = np.mean(tabtransformer_preds == stacking_preds)
stacking_itransformer_agreement = np.mean(itransformer_preds == stacking_preds)

if stacking_tab_agreement > 0.5 and stacking_itransformer_agreement > 0.1:
    print(f"✅ 스태킹 앙상블 추천")
    print(f"   → 두 모델의 장점을 균형있게 활용")
    print(f"   → 자동으로 최적 가중치 학습")
    print(f"   → 제출 파일: stacking_ensemble_submission.csv")
elif stacking_tab_agreement > 0.8:
    print(f"⚠️  스태킹이 TabTransformer와 너무 유사")
    print(f"   → 단순 가중 평균 사용 고려")
else:
    print(f"💡 스태킹과 다른 방법들을 모두 시도해보세요")
    print(f"   → 다양한 앙상블 방법 실험 권장")

print(f"\n생성된 파일들:")
print(f"  - stacking_ensemble_submission.csv (제출용)")
print(f"  - stacking_ensemble_detailed.csv (상세 결과)")


In [9]:
# TabTransformer + iTransformer 소프트보팅 앙상블
print("=" * 60)
print("TabTransformer + iTransformer 소프트보팅 앙상블")
print("=" * 60)

import pandas as pd
import numpy as np
import os

# 1. 데이터 로드
print("\n1. 데이터 로드")
print("-" * 30)

# TabTransformer 5폴드 결과 로드
if os.path.exists("tabtransformer_5fold_detailed.csv"):
    print("✅ TabTransformer 5폴드 결과 로드")
    tabtransformer_df = pd.read_csv("tabtransformer_5fold_detailed.csv")
    tabtransformer_probs = tabtransformer_df[[f'prob_{i}' for i in range(21)]].values
    tabtransformer_preds = tabtransformer_df['target'].values
    test_ids = tabtransformer_df['ID'].values
    print(f"TabTransformer 데이터 형태: {tabtransformer_probs.shape}")
else:
    print("❌ TabTransformer 5폴드 결과를 찾을 수 없습니다.")
    exit()

# iTransformer CV 앙상블 결과 로드
if os.path.exists("itransformer_cv_ensemble_detailed.csv"):
    print("✅ iTransformer CV 앙상블 결과 로드")
    itransformer_df = pd.read_csv("itransformer_cv_ensemble_detailed.csv")
    itransformer_probs = itransformer_df[[f'prob_{i}' for i in range(21)]].values
    itransformer_preds = itransformer_df['target'].values
    print(f"iTransformer 데이터 형태: {itransformer_probs.shape}")
else:
    print("❌ iTransformer CV 앙상블 결과를 찾을 수 없습니다.")
    exit()

# 2. 다양한 소프트보팅 가중치 실험
print("\n2. 소프트보팅 가중치 실험")
print("-" * 30)

# 다양한 가중치 조합 실험
weight_combinations = [
    ("동일 가중치", 0.5, 0.5),
    ("TabTransformer 우세", 0.7, 0.3),
    ("iTransformer 우세", 0.3, 0.7),
    ("TabTransformer 강우세", 0.9, 0.1),
    ("iTransformer 강우세", 0.1, 0.9),
    ("TabTransformer 극우세", 0.95, 0.05),
    ("iTransformer 극우세", 0.05, 0.95),
]

soft_voting_results = []

for method_name, tab_weight, itransformer_weight in weight_combinations:
    # 소프트보팅: 가중 평균
    ensemble_probs = tab_weight * tabtransformer_probs + itransformer_weight * itransformer_probs
    ensemble_preds = np.argmax(ensemble_probs, axis=1)
    
    # 성능 지표 계산
    tab_agreement = np.mean(tabtransformer_preds == ensemble_preds)
    itransformer_agreement = np.mean(itransformer_preds == ensemble_preds)
    avg_agreement = (tab_agreement + itransformer_agreement) / 2
    
    # 엔트로피 계산
    entropy = -np.sum(ensemble_probs * np.log(ensemble_probs + 1e-8), axis=1).mean()
    
    # 예측 분포 균등성
    pred_distribution = np.bincount(ensemble_preds, minlength=21)
    distribution_std = np.std(pred_distribution)
    
    soft_voting_results.append({
        'method': method_name,
        'tab_weight': tab_weight,
        'itransformer_weight': itransformer_weight,
        'tab_agreement': tab_agreement,
        'itransformer_agreement': itransformer_agreement,
        'avg_agreement': avg_agreement,
        'entropy': entropy,
        'distribution_std': distribution_std,
        'predictions': ensemble_preds,
        'probabilities': ensemble_probs
    })
    
    print(f"{method_name:20s}: 가중치({tab_weight:.2f}, {itransformer_weight:.2f}) | "
          f"Tab일치도={tab_agreement:.4f} | iT일치도={itransformer_agreement:.4f} | "
          f"평균={avg_agreement:.4f} | 엔트로피={entropy:.4f}")

# 3. 최적 소프트보팅 방법 선택
print("\n3. 최적 소프트보팅 방법 선택")
print("-" * 30)

# 다양한 기준으로 최적 방법 선택
criteria = [
    ("평균 일치도 최대", lambda x: x['avg_agreement']),
    ("엔트로피 최소", lambda x: -x['entropy']),  # 엔트로피가 낮을수록 좋음
    ("분포 균등성 최대", lambda x: -x['distribution_std']),  # 표준편차가 낮을수록 균등
    ("TabTransformer 일치도 최대", lambda x: x['tab_agreement']),
    ("iTransformer 일치도 최대", lambda x: x['itransformer_agreement']),
]

print("기준별 최적 방법:")
print("-" * 50)

for criterion_name, criterion_func in criteria:
    best_result = max(soft_voting_results, key=criterion_func)
    print(f"{criterion_name:25s}: {best_result['method']:20s} "
          f"(값: {criterion_func(best_result):.4f})")

# 4. 종합 평가로 최종 선택
print("\n4. 종합 평가")
print("-" * 30)

# 각 방법에 대해 종합 점수 계산 (정규화된 점수들의 평균)
def calculate_composite_score(result):
    # 각 지표를 0-1로 정규화
    max_avg_agreement = max(r['avg_agreement'] for r in soft_voting_results)
    min_entropy = min(r['entropy'] for r in soft_voting_results)
    max_entropy = max(r['entropy'] for r in soft_voting_results)
    min_distribution_std = min(r['distribution_std'] for r in soft_voting_results)
    max_distribution_std = max(r['distribution_std'] for r in soft_voting_results)
    
    # 정규화된 점수들
    agreement_score = result['avg_agreement'] / max_avg_agreement
    entropy_score = (max_entropy - result['entropy']) / (max_entropy - min_entropy) if max_entropy > min_entropy else 1.0
    distribution_score = (max_distribution_std - result['distribution_std']) / (max_distribution_std - min_distribution_std) if max_distribution_std > min_distribution_std else 1.0
    
    # 종합 점수 (가중 평균)
    composite_score = 0.4 * agreement_score + 0.3 * entropy_score + 0.3 * distribution_score
    return composite_score

# 각 방법의 종합 점수 계산
for result in soft_voting_results:
    result['composite_score'] = calculate_composite_score(result)

# 종합 점수 기준으로 정렬
soft_voting_results.sort(key=lambda x: x['composite_score'], reverse=True)

print("종합 점수 기준 순위:")
print("-" * 50)
for i, result in enumerate(soft_voting_results, 1):
    print(f"{i:2d}. {result['method']:20s}: {result['composite_score']:.4f} "
          f"(일치도: {result['avg_agreement']:.4f}, 엔트로피: {result['entropy']:.4f})")

# 최적 방법 선택
best_soft_voting = soft_voting_results[0]
print(f"\n✅ 최적 소프트보팅 방법: {best_soft_voting['method']}")
print(f"   가중치: TabTransformer={best_soft_voting['tab_weight']:.2f}, iTransformer={best_soft_voting['itransformer_weight']:.2f}")
print(f"   종합 점수: {best_soft_voting['composite_score']:.4f}")

# 5. 최종 결과 생성
print("\n5. 최종 결과 생성")
print("-" * 30)

final_preds = best_soft_voting['predictions']
final_probs = best_soft_voting['probabilities']

# 소프트보팅 제출 파일
soft_voting_submission = pd.DataFrame({
    "ID": test_ids,
    "target": final_preds
})
soft_voting_submission.to_csv("soft_voting_ensemble_submission.csv", index=False)

# 소프트보팅 상세 결과
soft_voting_detailed = pd.DataFrame({
    "ID": test_ids,
    "target": final_preds,
    **{f"prob_{i}": final_probs[:, i] for i in range(21)}
})
soft_voting_detailed.to_csv("soft_voting_ensemble_detailed.csv", index=False)

print(f"✅ 소프트보팅 앙상블 완료!")
print(f"제출 파일: soft_voting_ensemble_submission.csv")
print(f"상세 결과: soft_voting_ensemble_detailed.csv")
print(f"최적 방법: {best_soft_voting['method']}")
print(f"예측 분포: {np.bincount(final_preds, minlength=21)}")


TabTransformer + iTransformer 소프트보팅 앙상블

1. 데이터 로드
------------------------------
✅ TabTransformer 5폴드 결과 로드
TabTransformer 데이터 형태: (15004, 21)
✅ iTransformer CV 앙상블 결과 로드
iTransformer 데이터 형태: (15004, 21)

2. 소프트보팅 가중치 실험
------------------------------
동일 가중치              : 가중치(0.50, 0.50) | Tab일치도=0.9041 | iT일치도=0.9875 | 평균=0.9458 | 엔트로피=0.6542
TabTransformer 우세   : 가중치(0.70, 0.30) | Tab일치도=0.9210 | iT일치도=0.9697 | 평균=0.9454 | 엔트로피=0.7930
iTransformer 우세     : 가중치(0.30, 0.70) | Tab일치도=0.8974 | iT일치도=0.9949 | 평균=0.9462 | 엔트로피=0.4958
TabTransformer 강우세  : 가중치(0.90, 0.10) | Tab일치도=0.9539 | iT일치도=0.9369 | 평균=0.9454 | 엔트로피=0.9151
iTransformer 강우세    : 가중치(0.10, 0.90) | Tab일치도=0.8942 | iT일치도=0.9986 | 평균=0.9464 | 엔트로피=0.3082
TabTransformer 극우세  : 가중치(0.95, 0.05) | Tab일치도=0.9700 | iT일치도=0.9214 | 평균=0.9457 | 엔트로피=0.9431
iTransformer 극우세    : 가중치(0.05, 0.95) | Tab일치도=0.8937 | iT일치도=0.9993 | 평균=0.9465 | 엔트로피=0.2530

3. 최적 소프트보팅 방법 선택
------------------------------
기준별 최적 방법:
---------------------

In [10]:
# 소프트보팅 vs 스태킹 vs 하드보팅 비교 분석
print("=" * 60)
print("앙상블 방법별 종합 비교 분석")
print("=" * 60)

# 1. 모든 앙상블 방법 결과 로드
print("\n1. 앙상블 방법별 결과 로드")
print("-" * 30)

ensemble_methods = {}

# 소프트보팅 (방금 생성한 것)
if 'best_soft_voting' in locals():
    ensemble_methods['소프트보팅'] = {
        'predictions': best_soft_voting['predictions'],
        'probabilities': best_soft_voting['probabilities'],
        'method_name': best_soft_voting['method']
    }
    print("✅ 소프트보팅 결과 로드")

# 스태킹 (이전 셀에서 생성한 것)
if 'stacking_preds' in locals() and 'stacking_probs' in locals():
    ensemble_methods['스태킹'] = {
        'predictions': stacking_preds,
        'probabilities': stacking_probs,
        'method_name': 'LogisticRegression 스태킹'
    }
    print("✅ 스태킹 결과 로드")

# 하드보팅 (이전 셀에서 생성한 것)
if 'hard_voting_preds' in locals():
    # 하드보팅은 확률이 없으므로 예측만 저장
    ensemble_methods['하드보팅'] = {
        'predictions': hard_voting_preds,
        'probabilities': None,
        'method_name': '하드보팅 (확신도 기반)'
    }
    print("✅ 하드보팅 결과 로드")

# 2. 각 방법의 성능 지표 계산
print("\n2. 성능 지표 계산")
print("-" * 30)

comparison_results = []

for method_name, result in ensemble_methods.items():
    preds = result['predictions']
    probs = result['probabilities']
    
    # 기본 지표
    tab_agreement = np.mean(tabtransformer_preds == preds)
    itransformer_agreement = np.mean(itransformer_preds == preds)
    avg_agreement = (tab_agreement + itransformer_agreement) / 2
    
    # 예측 분포
    pred_distribution = np.bincount(preds, minlength=21)
    distribution_std = np.std(pred_distribution)
    
    # 엔트로피 (확률이 있는 경우만)
    if probs is not None:
        entropy = -np.sum(probs * np.log(probs + 1e-8), axis=1).mean()
    else:
        entropy = None
    
    comparison_results.append({
        'method': method_name,
        'tab_agreement': tab_agreement,
        'itransformer_agreement': itransformer_agreement,
        'avg_agreement': avg_agreement,
        'entropy': entropy,
        'distribution_std': distribution_std,
        'pred_distribution': pred_distribution
    })

# 3. 결과 비교 테이블
print("\n3. 앙상블 방법별 성능 비교")
print("-" * 80)

print(f"{'방법':12s} {'Tab일치도':>10s} {'iT일치도':>10s} {'평균일치도':>10s} {'엔트로피':>10s} {'분포균등성':>10s}")
print("-" * 80)

for result in comparison_results:
    entropy_str = f"{result['entropy']:.4f}" if result['entropy'] is not None else "N/A"
    print(f"{result['method']:12s} {result['tab_agreement']:>10.4f} {result['itransformer_agreement']:>10.4f} "
          f"{result['avg_agreement']:>10.4f} {entropy_str:>10s} {result['distribution_std']:>10.2f}")

# 4. 각 방법의 특징 분석
print("\n4. 각 방법의 특징 분석")
print("-" * 30)

for result in comparison_results:
    method = result['method']
    print(f"\n{method}:")
    print(f"  - TabTransformer 일치도: {result['tab_agreement']:.4f}")
    print(f"  - iTransformer 일치도: {result['itransformer_agreement']:.4f}")
    print(f"  - 평균 일치도: {result['avg_agreement']:.4f}")
    if result['entropy'] is not None:
        print(f"  - 엔트로피: {result['entropy']:.4f}")
    print(f"  - 예측 분포 표준편차: {result['distribution_std']:.2f}")
    print(f"  - 예측 분포: {result['pred_distribution']}")

# 5. 최적 방법 추천
print("\n5. 최적 방법 추천")
print("-" * 30)

# 각 기준별 최적 방법 찾기
criteria_analysis = {
    'TabTransformer 일치도': max(comparison_results, key=lambda x: x['tab_agreement']),
    'iTransformer 일치도': max(comparison_results, key=lambda x: x['itransformer_agreement']),
    '평균 일치도': max(comparison_results, key=lambda x: x['avg_agreement']),
    '분포 균등성': min(comparison_results, key=lambda x: x['distribution_std'])
}

print("기준별 최적 방법:")
for criterion, best_result in criteria_analysis.items():
    print(f"  {criterion}: {best_result['method']}")

# 엔트로피가 있는 방법들만 엔트로피 기준 분석
entropy_results = [r for r in comparison_results if r['entropy'] is not None]
if entropy_results:
    best_entropy = min(entropy_results, key=lambda x: x['entropy'])
    print(f"  엔트로피 최소: {best_entropy['method']}")

# 6. 종합 추천
print("\n6. 종합 추천")
print("-" * 30)

# 평균 일치도가 가장 높은 방법
best_agreement = max(comparison_results, key=lambda x: x['avg_agreement'])

print(f"✅ 추천 방법: {best_agreement['method']}")
print(f"   이유: 평균 일치도가 가장 높음 ({best_agreement['avg_agreement']:.4f})")

# 각 방법의 장단점
print(f"\n각 방법의 장단점:")
print(f"  소프트보팅:")
print(f"    장점: 간단하고 직관적, 가중치 조정 가능")
print(f"    단점: 수동으로 가중치 설정 필요")
print(f"  스태킹:")
print(f"    장점: 자동으로 최적 가중치 학습, 더 정교한 조합")
print(f"    단점: 복잡하고 해석이 어려움")
print(f"  하드보팅:")
print(f"    장점: 가장 간단하고 해석하기 쉬움")
print(f"    단점: 확률 정보 활용 불가, 극단적 선택 가능")

# 7. 최종 제출 파일 정리
print("\n7. 생성된 제출 파일들")
print("-" * 30)

submission_files = [
    "soft_voting_ensemble_submission.csv",
    "stacking_ensemble_submission.csv", 
    "hard_voting_submission.csv"
]

print("사용 가능한 제출 파일들:")
for file in submission_files:
    if os.path.exists(file):
        print(f"  ✅ {file}")
    else:
        print(f"  ❌ {file} (생성되지 않음)")

print(f"\n최종 권장사항:")
print(f"  1. {best_agreement['method']} 방법을 우선 시도")
print(f"  2. 여러 방법의 결과를 비교하여 최적 선택")
print(f"  3. 실제 대회에서는 여러 방법을 모두 제출해볼 수 있음")


앙상블 방법별 종합 비교 분석

1. 앙상블 방법별 결과 로드
------------------------------
✅ 소프트보팅 결과 로드
✅ 하드보팅 결과 로드

2. 성능 지표 계산
------------------------------

3. 앙상블 방법별 성능 비교
--------------------------------------------------------------------------------
방법               Tab일치도      iT일치도      평균일치도       엔트로피      분포균등성
--------------------------------------------------------------------------------
소프트보팅            0.8937     0.9993     0.9465     0.2530     137.64
하드보팅             0.1012     0.1027     0.1019        N/A      30.65

4. 각 방법의 특징 분석
------------------------------

소프트보팅:
  - TabTransformer 일치도: 0.8937
  - iTransformer 일치도: 0.9993
  - 평균 일치도: 0.9465
  - 엔트로피: 0.2530
  - 예측 분포 표준편차: 137.64
  - 예측 분포: [ 763  673  420  873  740  472  714  714 1045  608  680  669  953  706
  709  878  637  714  702  621  713]

하드보팅:
  - TabTransformer 일치도: 0.1012
  - iTransformer 일치도: 0.1027
  - 평균 일치도: 0.1019
  - 예측 분포 표준편차: 30.65
  - 예측 분포: [678 682 704 716 708 696 746 737 762 720 710 711 707 793 730 649 69